<a href="https://colab.research.google.com/github/Akpati-Lucan/algoverse-research/blob/master/Hebbian_DIffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load all libraries that will be used in this notebook


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import numpy as np
import time

# Primitive Hebbian Learning Model

First we create a primitive Hebbian Learning Nueral Network at test out it's perfomance
Then we create a train function to train the model Finally we add a method predictor to predict the output given a certain input vector

In [ ]:
class HebbianNetwork(nn.Module):

    def __init__(self, input_size, output_size, learning_rate=0.01):
        super().__init__()

        self.weights = torch.rand(input_size, output_size)
        self.learning_rate = learning_rate

    def train_hebbian(self, inputs, outputs):
        for x, y in zip(inputs, outputs):
            update = torch.outer(x, y)
            self.weights += self.learning_rate * update

    def predict(self, x):
        return torch.matmul(x, self.weights)

We then create a function to measure how good our network is
that is measuring the time it took to train for a certain number of epochs, and how accurate the network is.

In [ ]:
def evaluate_network(network, inputs, labels, epochs):
    start_time = time.time()

    # Training phase
    for epoch in range(epochs):
        network.train(inputs, labels)

    end_time = time.time()

    training_time = end_time - start_time

    # Testing phase
    correct = 0

    for i in range(len(inputs)):
        prediction = network.predict(inputs[i])

        # Convert prediction into class index
        predicted_class = np.argmax(prediction)
        true_class = np.argmax(labels[i])

        if predicted_class == true_class:
            correct += 1

    accuracy = correct / len(inputs)

    return training_time, accuracy

Setup data that will be used to train and validate networks

In [ ]:
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ]),
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True)
    ]),
)

batch_size = 64

train_dataloader = DataLoader(
    training_data,
    batch_size=batch_size,
    shuffle=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
# Data Check
for X, y in train_dataloader:
    print(X.shape)
    print(y.shape)
    break